# Phase 2 — Negative Review Classification

This notebook tests a simple hypothesis: **among orders that were delivered late, can Logistic Regression predict which customers are likely to give a negative review (1–2 stars)?**

The workflow is intentionally simple and aligned to the project scope: prepare the data, split into training and test sets, fit one Logistic Regression model, evaluate it, interpret it, and save it for Phase 3.


## 1. Define the problem

**Target**
* `1`: review score 1–2 (negative review)
* `0`: review score 3–5

**Population:** delivered orders that were late and subsequently received a review.

**Prediction point:** immediately after the late delivery is completed, before the customer's review is known.

Phase 1 defines `days_early` so negative values mean late. For example, `-10` means 10 days late.

Review IDs, review text and review timestamps are excluded because they are only known after the outcome. Phase 2 reuses the cleaned Phase 1 analytical tables.


### Load the Phase 1 data

The notebook uses `data/processed/order_level.csv` and `data/processed/item_level.csv`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification_data import (
    MODEL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_classification_dataset,
    split_features_target,
)
from src.classification_evaluation import classification_metrics
from src.classification_model import (
    build_logistic_pipeline,
    predict_from_artifact,
    save_inference_artifact,
)

RANDOM_SEED = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

order_level = pd.read_csv(PROCESSED_DIR / "order_level.csv")
item_level = pd.read_csv(PROCESSED_DIR / "item_level.csv")

print(f"order_level: {order_level.shape}")
print(f"item_level: {item_level.shape}")


### Build the modelling dataset

Create one row per eligible late order and inspect the target balance. The reusable data preparation remains in `src/classification_data.py`.


In [ ]:
classification_data = build_classification_dataset(order_level, item_level)

print(f"Rows available for modelling: {len(classification_data):,}")
display(
    classification_data[TARGET_COLUMN]
    .value_counts()
    .rename(index={0: "Review 3–5", 1: "Negative review 1–2"})
    .rename("orders")
    .to_frame()
)

feature_audit = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "type": ["numeric" if name in NUMERIC_FEATURES else "categorical" for name in MODEL_FEATURES],
})
display(feature_audit)

assert classification_data["order_id"].is_unique
assert not any(name.startswith("review_") for name in MODEL_FEATURES)
assert "order_id" not in MODEL_FEATURES


## 2. Split the data

Use 80% of the data for training and keep 20% aside for the final test. Stratification keeps the proportion of negative reviews similar in both sets.


In [ ]:
X, y = split_features_target(classification_data)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Negative-review rate in training: {y_train.mean():.1%}")
print(f"Negative-review rate in test: {y_test.mean():.1%}")


## 3. Train Logistic Regression

The scikit-learn Pipeline keeps preprocessing and Logistic Regression together. Numeric values are imputed and standardised; categorical values are imputed and one-hot encoded.

The project requires cross-validation, so we use **5-fold cross-validation** as a simple check of how consistently the model performs on different parts of the training data. We report the five F1 scores and their average. There is no model tournament and no threshold tuning.


In [ ]:
model = build_logistic_pipeline(random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")

print("5-fold F1 scores:", np.round(cv_f1_scores, 3))
print(f"Mean F1: {cv_f1_scores.mean():.3f}")

model.fit(X_train, y_train)


## 4. Evaluate on the test set

Use the fitted model once on the held-out test set. Logistic Regression's standard **0.50 probability cutoff** is kept unchanged: probability at or above 0.50 is predicted as a negative review.

Precision, Recall, F1, PR-AUC and ROC-AUC are reported because they are required classification measures for the project. Balanced accuracy is also shown as a useful summary when the two classes are not equally common.


In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
test_metrics = classification_metrics(y_test, test_probability, threshold=0.50)

display(pd.Series(test_metrics, name="test_set").round(3))

test_prediction = (test_probability >= 0.50).astype(int)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["Review 3–5", "Negative review 1–2"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_prediction,
    display_labels=["Review 3–5", "Negative 1–2"],
)


## 5. Interpret the model

Logistic Regression coefficients show which inputs are associated with higher or lower odds of a negative review while the other model variables are held fixed.

* Positive coefficient: associated with higher odds of a negative review.
* Negative coefficient: associated with lower odds.
* Odds ratio above 1: higher odds.
* Odds ratio below 1: lower odds.

These are **associations, not causal effects**.


In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})
coefficient_table["odds_ratio"] = np.exp(coefficient_table["coefficient"])

strongest_associations = pd.concat([
    coefficient_table.nsmallest(10, "coefficient"),
    coefficient_table.nlargest(10, "coefficient"),
]).drop_duplicates()

display(
    strongest_associations
    .sort_values("coefficient")
    .reset_index(drop=True)
    .round(3)
)


## 6. Save the model for Phase 3

Save the fitted preprocessing + Logistic Regression Pipeline as one `.pkl` file. Phase 3 FastAPI will load this file and use it for predictions. The model uses the standard 0.50 cutoff.


In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / "deployment" / "negative_review_logistic_pipeline.pkl"

save_inference_artifact(
    model,
    str(ARTIFACT_PATH),
    threshold=0.50,
    metadata={
        "training_data_contract": "Phase 1 order_level.csv + item_level.csv",
        "model": "Logistic Regression",
        "classification_threshold": 0.50,
    },
)

import joblib

artifact = joblib.load(ARTIFACT_PATH)
display(predict_from_artifact(artifact, X_test.iloc[[0]]))
print(f"Saved model artifact: {ARTIFACT_PATH}")
